## Convex hull and sphericity measurements

In [2]:

"""
3D Image Analysis Pipeline for Measuring Biological Components

This notebook processes 3D biological image data (stored in HDF5 files) to:
1. Label connected components in binary masks
2. Filter components by volume
3. Calculate convex hull properties and sphericity
4. Save filtered results and measurements

"""

# Library imports
import h5py
import numpy as np
import pandas as pd
import tifffile as tiff
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull
from skimage.measure import label, regionprops
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import math

### Processing Functions

In [3]:
def measure_components(binary_mask):
    """
    Label connected components in a 3D binary mask and measure their properties.
    
    Args:
        binary_mask (ndarray): 3D binary array where 1 represents foreground
        
    Returns:
        DataFrame: Measurements for each component (volume, centroid)
        ndarray: Labeled array where each component has a unique integer value
    """
    labeled_array, num_features = label(binary_mask, return_num=True)
    props = regionprops(labeled_array)
   
    data = {
        'Component': [],
        'Volume': [],
        'Centroid': []
    }

    for i, prop in enumerate(props):
        volume = prop.area  
        centroid = prop.centroid 
       
        data['Component'].append(i + 1)
        data['Volume'].append(volume)
        data['Centroid'].append(centroid)
   
    df = pd.DataFrame(data)
   
    return df, labeled_array


In [4]:
def filter_by_volume(df, labeled_array, volume_threshold, output_path):
    """
    Filter components based on volume threshold and save the result.
    
    Args:
        df (DataFrame): Component measurements from measure_components()
        labeled_array (ndarray): Labeled component array
        volume_threshold (int): Minimum volume to keep
        output_path (str): Path to save filtered TIFF file
        
    Returns:
        DataFrame: Filtered component measurements
        ndarray: Filtered labeled array
    """
    filtered_df = df[df['Volume'] > volume_threshold]
   
    filtered_labeled_array = np.zeros_like(labeled_array)
    for component in filtered_df['Component']:
        filtered_labeled_array[labeled_array == component] = component
   
    tiff.imwrite(output_path, filtered_labeled_array.astype(np.uint16))
   
    return filtered_df, filtered_labeled_array


In [5]:
def plot_convex_hull_of_points(points, volume, sphericity, plot_folder, component):
    """
    Create and save a 3D plot of points with their convex hull.
    
    Args:
        points (ndarray): 3D coordinates of component voxels
        volume (float): Volume of the component
        sphericity (float): Calculated sphericity (0-1)
        plot_folder (str): Directory to save plots
        component (int): Component identifier
    """
    hull = ConvexHull(points)
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    ax.scatter(points[:, 0], points[:, 1], points[:, 2], 'o')
    ax.set_title(f'Component {component}\nVolume: {volume:.2f}, Sphericity: {sphericity:.2f}')
    
    vertices = [points[face] for face in hull.simplices]
    ax.add_collection3d(Poly3DCollection(vertices, facecolors='cyan', linewidths=1, edgecolors='r'))
    
    plot_path = os.path.join(plot_folder, f'component_{component}_convex_hull.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()


In [6]:
def convex_hull(labeled_array, output_csv_path, volumes_csv_path, plot_folder, sphericity_threshold=0.5):
    """
    Calculate convex hull properties for each component and save results.
    
    Args:
        labeled_array (ndarray): Labeled component array
        output_csv_path (str): Path to save hull point coordinates
        volumes_csv_path (str): Path to save component measurements
        plot_folder (str): Directory to save visualization plots
        sphericity_threshold (float): Minimum sphericity to keep (0-1)
    """
    unique_components = np.unique(labeled_array)
    unique_components = unique_components[unique_components != 0]  

    all_hull_points = []
    volumes = []
    sphericities = []
    valid_components = []
    centroids = []
    
    os.makedirs(plot_folder, exist_ok=True)
    
    for component in unique_components:
        print(f"Processing component {component} of {len(unique_components)}")
        mask = (labeled_array == component)
        points = np.argwhere(mask)
        
        # Skip components with too few points for convex hull
        if points.shape[0] < 4:
            continue

        hull = ConvexHull(points)
        volume = hull.volume
        surface_area = hull.area
        sphericity = (36 * np.pi * (volume**2)) / (surface_area**3)
        
        # Skip components below sphericity threshold
        if sphericity < sphericity_threshold:
            continue
        
        all_hull_points.extend([[component, *points[vertex]] for vertex in hull.vertices])
        volumes.append(volume)
        sphericities.append(sphericity)
        valid_components.append(component)
        
        centroid = np.mean(points[hull.vertices], axis=0)
        centroids.append(centroid)
    
    if len(valid_components) > 0:
        hull_df = pd.DataFrame(all_hull_points, columns=['Component', 'X', 'Y', 'Z'])
        hull_df.to_csv(output_csv_path, index=False)

        volumes_df = pd.DataFrame({
            'Component': valid_components, 
            'Volume': volumes, 
            'Sphericity': sphericities,
            'Centroid_X': [c[0] for c in centroids],
            'Centroid_Y': [c[1] for c in centroids],
            'Centroid_Z': [c[2] for c in centroids]
        })
        volumes_df.to_csv(volumes_csv_path, index=False)
    else:
        print("Warning: No components met the sphericity threshold criteria")



### Main Processing Pipeline

In [7]:

def process_files_in_folder(input_folder, output_folder, volume_threshold, sphericity_threshold=0.5):
    """
    Process all HDF5 files in a folder through the analysis pipeline.
    
    Args:
        input_folder (str): Directory containing HDF5 files
        output_folder (str): Directory to save results
        volume_threshold (int): Minimum component volume to analyze
        sphericity_threshold (float): Minimum sphericity to keep (0-1)
    """
    
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    for filename in os.listdir(input_folder):
        print(filename)
        if filename.endswith('.h5'):  
            file_path = os.path.join(input_folder, filename)
            
            with h5py.File(file_path, 'r') as f:
                labeled_array = f['data'][:]
            
            base_name = filename.split('_')[0]
            
            base_output_folder = os.path.join(output_folder, base_name)
            if not os.path.exists(base_output_folder):
                os.makedirs(base_output_folder)
            
            output_csv_path = os.path.join(base_output_folder, f'{base_name}_convex_hull.csv')
            volumes_csv_path = os.path.join(base_output_folder, f'{base_name}_volumes.csv')
            plot_folder = os.path.join(base_output_folder, f'{base_name}_convex_hull_plots')
            filtered_output_path = os.path.join(base_output_folder, f'{base_name}_filtered.tiff')
         
            df, labeled_array = measure_components(labeled_array)
          
            filtered_df, filtered_labeled_array = filter_by_volume(df, labeled_array, volume_threshold, filtered_output_path)
            
            convex_hull(filtered_labeled_array, output_csv_path, volumes_csv_path, plot_folder, sphericity_threshold)



### Execution Parameters

In [ ]:
# Input and output paths 
input_folder = '/sv_measurements/20240304/CG59slot1_SR-older/h5_seg/'
output_folder = os.path.join(input_folder, 'measurements2')

# Processing thresholds
volume_threshold = 8000  # Minimum volume in voxels
sphericity_threshold = 0.56  # Minimum sphericity (0-1)

# Run the pipeline
process_files_in_folder(input_folder, output_folder, volume_threshold, sphericity_threshold)